# Void Clipex bot — Kaggle runner (free, 12 hours)

**Before you press Run All**

1. Right panel -> **Settings** -> **Internet: On** (warna kuch download nahi hoga).
2. **Add-ons -> Secrets** -> do secrets banayein:
   - `TELEGRAM_BOT_TOKEN` = aap ka bot token (BotFather se)
   - `OPENROUTER_API_KEY` = `sk-or-v1-...` key
3. Phir **Run All**.

Bot start hone mein 5-8 minute lega. Uske baad Telegram par `hi` bhejein.
Cell 4 chalta rehna chahiye; tab band karne se session ruk sakta hai. Ek session ~12 ghante chalta hai.


In [ ]:
#@title 1) Install node, openclaw, ffmpeg tools (4-6 min)
import json as _json, os, subprocess, urllib.request

def sh(cmd, quiet=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet and r.returncode != 0:
        print("err:", cmd[:60], (r.stderr or r.stdout)[-300:])
    return r

print("ffmpeg:", sh("ffmpeg -version | head -1", quiet=True).stdout.strip()[:50])

if "v24" not in sh("node -v", quiet=True).stdout:
    idx = _json.loads(urllib.request.urlopen("https://nodejs.org/dist/index.json").read())
    ver = [x["version"] for x in idx if x["version"].startswith("v24")][0]
    print("node version:", ver)
    sh("curl -fsSL https://nodejs.org/dist/" + ver + "/node-" + ver + "-linux-x64.tar.xz -o /tmp/node.tar.xz")
    sh("tar -xJf /tmp/node.tar.xz -C /usr/local --strip-components=1")

print("node:", sh("node -v", quiet=True).stdout.strip())
sh("npm i -g openclaw@latest --no-audit --no-fund 2>&1 | tail -3")
print("openclaw:", sh("openclaw --version 2>&1 | head -1", quiet=True).stdout.strip())
sh("pip install -q yt-dlp faster-whisper 2>&1 | tail -2")
print("yt-dlp:", sh("yt-dlp --version", quiet=True).stdout.strip())

In [ ]:
#@title 2) Bot files + keys
import json, os, subprocess

BUNDLE = "https://base44.app/api/apps/6afd5d7f94aa00438e512b21/files/mp/public/6afd5d7f94aa00438e512b21/a57c5e21a_openclaw-vm-bundletar.gz"
HOME = os.path.expanduser("~")

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(("ok  " if r.returncode == 0 else "err ") + cmd[:70])
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-400:])
    return r

run("curl -fsSL " + BUNDLE + " -o /tmp/b.tgz")
os.makedirs("/tmp/bx", exist_ok=True)
subprocess.run("tar -xzf /tmp/b.tgz -C /tmp/bx", shell=True)
run("mkdir -p " + HOME + "/.openclaw && cp -rf /tmp/bx/oc/. " + HOME + "/.openclaw/")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretClient
        return UserSecretClient().get_secret(name) or ""
    except Exception:
        pass
    try:
        import kaggle_secrets as _ks
        for attr in dir(_ks):
            if "Client" in attr:
                try:
                    return getattr(_ks, attr)().get_secret(name) or ""
                except Exception:
                    pass
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        pass
    return os.environ.get(name, "")

TOKEN = get_secret("TELEGRAM_BOT_TOKEN")
ORK = get_secret("OPENROUTER_API_KEY")
if not TOKEN:
    TOKEN = input("Bot token paste karein: ").strip()
if not ORK:
    ORK = input("OpenRouter key paste karein: ").strip()

order = [("nvidia/nemotron-3-ultra-550b-a55b:free",1000000),("nvidia/nemotron-3.5-super-vl:free",1000000),
         ("baidu/ernie-5.5-flash-preview:free",1024000),("particle-ai/particle-1:free",1048576),
         ("inclusionai/ling-3.2-flash:free",1048576),("google/gemma-4-31b-it:free",262144),
         ("thinkingmachines/inkling-2-small:free",1048576),("nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",256000),
         ("cohere/north-mini-code:free",256000),("dots-studio/dots-3-note-preview:free",512000),
         ("recursive/flint-v1.0:free",262144),("nvidia/nemotron-3.5-lightning:free",1000000),
         ("google/gemma-4-26b-a4b-it:free",262144),("liquid/lfm-3-1.7b:free",262144)]

p = os.path.join(HOME, ".openclaw", "openclaw.json")
cfg = json.load(open(p))
tg = cfg.setdefault("channels", {}).setdefault("telegram", {})
tg["botToken"] = TOKEN
tg["enabled"] = True

models = [{"id":m,"name":m,"contextWindow":ctx,"maxTokens":8192,"input":["text"],
           "cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0},"reasoning":False,
           "compat":{"supportsTools":True,"supportsTemperature":True,"supportsUsageInStreaming":True}} for m,ctx in order]
cfg.setdefault("models",{}).setdefault("providers",{})["openrouter"] = {
    "baseUrl":"https://openrouter.ai/api/v1","api":"openai-completions","apiKey":ORK,"models":models}
d = cfg.setdefault("agents",{}).setdefault("defaults",{})
d["models"] = {("openrouter/"+m):{} for m,_ in order}
d["model"] = {"primary":"openrouter/"+order[0][0],
              "fallbacks":["openrouter/"+m for m,_ in order[1:]]}
json.dump(cfg, open(p,"w"), indent=2)
print("token set:", bool(TOKEN), "| openrouter:", bool(ORK), "| models:", len(order))

In [ ]:
#@title 3) Sanity check
import subprocess
print(subprocess.run("openclaw models list --all | grep openrouter | head -5", shell=True,
                     capture_output=True, text=True).stdout)
print(subprocess.run("openclaw models list --all | grep -c openrouter/", shell=True,
                     capture_output=True, text=True).stdout.strip(), "openrouter models loaded")

In [ ]:
#@title 4) Run the bot (keep this cell running)
import subprocess, time, os

os.environ["HOME"] = os.path.expanduser("~")
print("Bot start ho raha hai... Telegram par hi bhejein (1-2 min).\n")

while True:
    p = subprocess.Popen(["openclaw", "gateway", "run", "--force"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("\n[bot] gateway exited, restarting in 10s")
    time.sleep(10)